In [9]:
import os
import pandas as pd

# ================================
# PATHS (your folders)
# ================================
base_path = r"C:\Users\ramya\OneDrive\Desktop\China_projects\Nursing_multimodal_proejct"

clinical_path = os.path.join(base_path, "mimic3d.csv")

image_dataset1_path = os.path.join(base_path, "pressure_ulcers_expanded")   # change folder name if needed
image_dataset2_path = os.path.join(base_path, "dataset")   # change folder name if needed


In [5]:
# ================================
# STEP 1: LOAD CLINICAL DATA
# ================================
df = pd.read_csv(clinical_path)

print("Clinical dataset shape:", df.shape)
print(df.head())

Clinical dataset shape: (58976, 28)
   hadm_id gender  age  LOSdays admit_type             admit_location  \
0   100001      F   35     6.17  EMERGENCY  CLINIC REFERRAL/PREMATURE   
1   100003      M   59     4.04  EMERGENCY       EMERGENCY ROOM ADMIT   
2   100006      F   48    12.04  EMERGENCY       EMERGENCY ROOM ADMIT   
3   100007      F   73     7.29  EMERGENCY       EMERGENCY ROOM ADMIT   
4   100009      M   60     4.88  EMERGENCY  TRANSFER FROM HOSP/EXTRAM   

            AdmitDiagnosis insurance           religion marital_status  ...  \
0    DIABETIC KETOACIDOSIS   Private  PROTESTANT QUAKER       DIVORCED  ...   
1           UPPER GI BLEED   Private      NOT SPECIFIED         SINGLE  ...   
2               COPD FLARE   Private      NOT SPECIFIED         SINGLE  ...   
3        BOWEL OBSTRUCTION   Private             JEWISH        MARRIED  ...   
4  CORONARY ARTERY DISEASE   Private           CATHOLIC        MARRIED  ...   

  NumMicroLabs  NumNotes  NumOutput  NumRx NumProc

In [29]:
# ================================
# CREATE RISK LABELS
# ================================
def create_risk_label(row):
    score = 0
    
    # Age factor
    if 'age' in df.columns:
        if row['age'] > 65:
            score += 1

    # Length of stay
    if 'LOSgroupNum' in df.columns:
        if row['LOSgroupNum'] > 2:
            score += 1

    # Interaction intensity
    if 'TotalNumInteract' in df.columns:
        if row['TotalNumInteract'] > df['TotalNumInteract'].median():
            score += 1

    # Convert to risk class
    if score >= 2:
        return 2   # High Risk
    elif score == 1:
        return 1   # Moderate Risk
    else:
        return 0   # Low Risk


# Apply labeling
df['risk_label'] = df.apply(create_risk_label, axis=1)

# ================================
# VERIFY
# ================================
print("\nClinical Risk Distribution:")
print(df['risk_label'].value_counts())

# ================================
# SAVE NEW FILE
# ================================
save_path = os.path.join(base_path, "mimic3d_labeled.csv")
df.to_csv(save_path, index=False)

print("\n✅ Saved labeled dataset to:", save_path)


Clinical Risk Distribution:
risk_label
1    24370
2    19326
0    15280
Name: count, dtype: int64

✅ Saved labeled dataset to: C:\Users\ramya\OneDrive\Desktop\China_projects\Nursing_multimodal_proejct\mimic3d_labeled.csv


In [22]:
label_map = {
    # Dataset1 (string folders)
    'stage_i': 0,
    'stage_ii': 1,
    'stage_iii': 2,
    'stage_iv': 2,
    'sdti': 2,
    'unstageable': 2,
    'invalid': -1,

    # Dataset2 (numeric folders)
    '1': 0,
    '2': 1,
    '3': 2,
    '4': 2
}

In [25]:
# =========================================
# NORMALIZATION FUNCTION (VERY IMPORTANT)
# =========================================
def normalize_label(name):
    return (
        name.strip()
            .lower()
            .replace(" ", "")
            .replace("-", "_")
    )


In [27]:
# =========================================
# LOAD IMAGE DATASET FUNCTION
# =========================================
def load_image_dataset(folder_path, dataset_name):
    data = []
    print(f"\n📂 Loading {dataset_name}...")

    for class_name in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_name)

        if not os.path.isdir(class_path):
            continue

        class_clean = normalize_label(class_name)

        if class_clean not in label_map:
            print(f"❌ Skipped: {class_name} → {class_clean}")
            continue
        else:
            print(f"✅ Loaded: {class_name} → {class_clean}")

        label = label_map[class_clean]

        if label == -1:
            print(f"🚫 Ignored invalid class: {class_name}")
            continue

        file_count = 0

        for img_name in os.listdir(class_path):

            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(class_path, img_name)

            data.append({
                'image_path': img_path,
                'original_label': class_clean,
                'risk_label': label
            })

            file_count += 1

        print(f"   → {file_count} images loaded")

    return pd.DataFrame(data)


# =========================================
# LOAD BOTH DATASETS
# =========================================
img_df1 = load_image_dataset(image_dataset1_path, "Dataset1 (Stage folders)")
img_df2 = load_image_dataset(image_dataset2_path, "Dataset2 (Numeric folders)")

# Combine
img_df = pd.concat([img_df1, img_df2], ignore_index=True)


# =========================================
# FINAL OUTPUT CHECK
# =========================================
print("\n==============================")
print("FINAL DATASET SUMMARY")
print("==============================")

print("\nTotal images:", len(img_df))

print("\nOriginal Label Distribution:")
print(img_df['original_label'].value_counts())

print("\nRisk Label Distribution:")
print(img_df['risk_label'].value_counts())




📂 Loading Dataset1 (Stage folders)...
✅ Loaded: Invalid → invalid
🚫 Ignored invalid class: Invalid
✅ Loaded: SDTI → sdti
   → 300 images loaded
✅ Loaded: Stage_I → stage_i
   → 284 images loaded
✅ Loaded: Stage_II → stage_ii
   → 379 images loaded
✅ Loaded: Stage_III → stage_iii
   → 345 images loaded
✅ Loaded: Stage_IV → stage_iv
   → 391 images loaded
✅ Loaded: Unstageable → unstageable
   → 300 images loaded

📂 Loading Dataset2 (Numeric folders)...
✅ Loaded: 1 → 1
   → 230 images loaded
✅ Loaded: 2 → 2
   → 313 images loaded
✅ Loaded: 3 → 3
   → 275 images loaded
✅ Loaded: 4 → 4
   → 273 images loaded

FINAL DATASET SUMMARY

Total images: 3090

Original Label Distribution:
original_label
stage_iv       391
stage_ii       379
stage_iii      345
2              313
sdti           300
unstageable    300
stage_i        284
3              275
4              273
1              230
Name: count, dtype: int64

Risk Label Distribution:
risk_label
2    1884
1     692
0     514
Name: count, dty

In [28]:
# =========================================
# SAVE FINAL DATA
# =========================================
save_path = os.path.join(base_path, "image_aligned.csv")
img_df.to_csv(save_path, index=False)

print("\n✅ Saved to:", save_path)


✅ Saved to: C:\Users\ramya\OneDrive\Desktop\China_projects\Nursing_multimodal_proejct\image_aligned.csv


In [30]:
# =========================================
# IMPORTS
# =========================================
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# =========================================
# PATHS
# =========================================
base_path = r"C:\Users\ramya\OneDrive\Desktop\China_projects\Nursing_multimodal_proejct"

clinical_path = os.path.join(base_path, "mimic3d_labeled.csv")
image_path = os.path.join(base_path, "image_aligned.csv")

In [31]:
# =========================================
# LOAD DATA
# =========================================
clinical_df = pd.read_csv(clinical_path)
image_df = pd.read_csv(image_path)

print("Clinical shape:", clinical_df.shape)
print("Image shape:", image_df.shape)

Clinical shape: (58976, 29)
Image shape: (3090, 3)


In [32]:
# =========================================
# MATCH DATA (IMPORTANT)
# =========================================
# Since datasets are unpaired → simulate pairing
min_len = min(len(clinical_df), len(image_df))

clinical_df = clinical_df.sample(min_len, random_state=42).reset_index(drop=True)
image_df = image_df.sample(min_len, random_state=42).reset_index(drop=True)

# Labels
y = clinical_df['risk_label']


In [33]:
# =========================================
# CLINICAL FEATURES
# =========================================
X_clinical = clinical_df.drop(columns=['risk_label'])

# Keep only numeric columns
X_clinical = X_clinical.select_dtypes(include=[np.number])

scaler = StandardScaler()
X_clinical = scaler.fit_transform(X_clinical)


In [34]:
# =========================================
# IMAGE LOADING FUNCTION
# =========================================
IMG_SIZE = 224

def load_images(paths):
    images = []
    for p in paths:
        try:
            img = load_img(p, target_size=(IMG_SIZE, IMG_SIZE))
            img = img_to_array(img) / 255.0
            images.append(img)
        except:
            images.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))
    return np.array(images)

X_image = load_images(image_df['image_path'])

In [35]:
# =========================================
# TRAIN TEST SPLIT
# =========================================
Xc_train, Xc_test, Xi_train, Xi_test, y_train, y_test = train_test_split(
    X_clinical, X_image, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [36]:
# =========================================
# CLASS WEIGHT (IMPORTANT)
# =========================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: 1.2580152671755724, 1: 0.8223552894211577, 2: 1.011042944785276}


In [37]:
# =========================================
# IMAGE MODEL (CNN)
# =========================================
image_input = Input(shape=(224,224,3))

base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=image_input)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)

image_features = x


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step


In [38]:
# =========================================
# CLINICAL MODEL (DNN)
# =========================================
clinical_input = Input(shape=(X_clinical.shape[1],))

c = Dense(128, activation='relu')(clinical_input)
c = BatchNormalization()(c)
c = Dropout(0.3)(c)
c = Dense(64, activation='relu')(c)

clinical_features = c

In [39]:
# =========================================
# FUSION LAYER
# =========================================
fusion = concatenate([image_features, clinical_features])

f = Dense(256, activation='relu')(fusion)
f = Dropout(0.4)(f)
f = Dense(128, activation='relu')(f)

output = Dense(3, activation='softmax')(f)

In [40]:
# =========================================
# FINAL MODEL
# =========================================
model = Model(inputs=[image_input, clinical_input], outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,239,043 (92.46 MB)

 Trainable params: 24,185,667 (92.26 MB)

 Non-trainable params: 53,376 (208.50 KB)

In [44]:
 #=========================================
# FIX DATA FORMAT
# =========================================
y_train = y_train.values
y_test = y_test.values

Xc_train = np.array(Xc_train)
Xc_test = np.array(Xc_test)

Xi_train = np.array(Xi_train)
Xi_test = np.array(Xi_test)

In [45]:
# =========================================
# TRAIN MODEL
# =========================================
history = model.fit(
    [Xi_train, Xc_train],
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights
)

Epoch 1/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 866s 13s/step - accuracy: 0.3699 - loss: 1.1675 - val_accuracy: 0.5152 - val_loss: 1.0740
Epoch 2/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 625s 10s/step - accuracy: 0.6092 - loss: 0.8228 - val_accuracy: 0.2949 - val_loss: 1.0842
Epoch 3/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 619s 10s/step - accuracy: 0.7101 - loss: 0.6365 - val_accuracy: 0.4727 - val_loss: 1.0111
Epoch 4/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 644s 10s/step - accuracy: 0.8064 - loss: 0.4736 - val_accuracy: 0.5657 - val_loss: 0.8247
Epoch 5/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 614s 10s/step - accuracy: 0.8615 - loss: 0.3628 - val_accuracy: 0.7172 - val_loss: 0.6928
Epoch 6/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 617s 10s/step - accuracy: 0.8681 - loss: 0.3245 - val_accuracy: 0.5980 - val_loss: 0.8312
Epoch 7/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 614s 10s/step - accuracy: 0.9177 - loss: 0.2266 - val_accuracy: 0.7212 - val_loss: 0.6338
Epoch 8/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 624s 10s/step - accuracy: 0.9079 - loss: 0.2223 - val_accuracy: 0.

In [46]:
# =========================================
# EVALUATION
# =========================================
loss, acc = model.evaluate([Xi_test, Xc_test], y_test)

print("\nTest Accuracy:", acc)

20/20 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.6134 - loss: 1.5188

Test Accuracy: 0.6375404596328735


In [47]:
# =========================================
# PREDICTIONS & METRICS
# =========================================
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict([Xi_test, Xc_test])
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))

20/20 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.80      0.77       164
           1       0.59      0.45      0.51       251
           2       0.60      0.74      0.66       203

    accuracy                           0.64       618
   macro avg       0.64      0.66      0.65       618
weighted avg       0.63      0.64      0.63       618


Confusion Matrix:
[[132  27   5]
 [ 44 112  95]
 [  3  50 150]]
